<a href="https://colab.research.google.com/github/Swarup215/Face-Tampering/blob/main/combined_deepfake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Combined Deepfake Detection Pipeline — 3-Dataset (Colab Pro)

**Datasets used:**
- `MyDrive/Dataset/DeepFake` → Real / Fake folders
- `MyDrive/Dataset/FaceForensic++` → original / DeepFake folders
- `MyDrive/Dataset/Celeb-DF.zip` → YouTube-real + Celeb-real + Celeb-synthesis

**Architecture:** MesoNet + EfficientNet-B4 + FFT Branch → Temporal Conv → BiLSTM

**Enhancements:** COBRA-inspired perceptual sharpening (preserves forensic fingerprints), robust face detection, fixed validation logic, per-epoch checkpoints, final weights saved as `combined.pth`

---

## ▶ Cells to run each session

### First time (full training):
Run ALL cells top-to-bottom (Ctrl+F9).

### Subsequent sessions (inference only — skip re-training):
1. Cell 1 — Install deps
2. Cell 3 — Imports & Config
3. Cell 4 — Mount Drive
4. Cell 17 — Model definition
5. Cell 18 — Load `combined.pth`
6. Cell 21 — Inference / predict_video()


In [ ]:
# ============================================================
# CELL 1 — FIXED Stable Installation (Colab)
# ============================================================

# Remove problematic versions
!pip uninstall -y torch torchvision torchaudio numpy protobuf mediapipe opencv-python -q

# Install compatible versions (VERY IMPORTANT)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

!pip install -q \
numpy==1.26.4 \
protobuf==4.25.3 \
mediapipe==0.10.20 \
opencv-python==4.9.0.80 \
scikit-learn \
pandas \
tqdm \
efficientnet_pytorch

print("✅ Clean stable environment ready")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, which is not installed.
tensorflow-hub 0.16.1 requires protobuf>=3.19.6, which is not installed.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, which is not installed.
orbax-checkpoint 0.11.33 requires protobuf, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
yfinance 0.2.66 requires protobuf>=3.19.0, which is not installed.
tensorboard 2.19.0 requires protobuf!=4.24.0,>=3.19.6, which is not installed.
tensorflow-datasets 4.9.9 requires protobuf>=3.20, which is not installed.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, which is not installed.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
numba 0.60.0 requires n

In [ ]:
# ============================================================
# FIX: Reset NumPy + Pandas compatibility
# ============================================================

!pip uninstall -y numpy pandas scikit-learn -q

!pip install -q numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2

print("✅ Fixed numpy-pandas compatibility")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
import os, shutil

mount_path = "/content/drive"
if os.path.islink(mount_path):
    os.unlink(mount_path)
elif os.path.isdir(mount_path):
    pass  # already a directory

os.makedirs(mount_path, exist_ok=True)
drive.mount(mount_path, force_remount=True)
print("Drive mounted.")


Mounted at /content/drive
Drive mounted.


In [ ]:
# ============================================================
# CELL 3 — Imports & Global Configuration
# ============================================================
import os, cv2, math, glob, json, time, random, shutil, zipfile, urllib.request
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix,
                              roc_auc_score, precision_score, recall_score)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from efficientnet_pytorch import EfficientNet

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── Dataset Paths ────────────────────────────────────────────
DS1_REAL = "/content/drive/MyDrive/Dataset/DeepFake/Real"
DS1_FAKE = "/content/drive/MyDrive/Dataset/DeepFake/Fake"

DS2_REAL = "/content/drive/MyDrive/Dataset/FaceForensic++/original"
DS2_FAKE = "/content/drive/MyDrive/Dataset/FaceForensic++/DeepFake"

CELEBDF_ZIP = "/content/drive/MyDrive/Dataset/Celeb-DF.zip"
CELEBDF_EXTRACT = "/content/Celeb_DF_extracted"

# ── Output Paths ─────────────────────────────────────────────
FRAMES_DIR       = "/content/deepfake_frames"
META_DIR         = "/content/deepfake_meta"
CHECKPOINT_ROOT  = "/content/drive/MyDrive/DeepfakeCheckpoints"
FINAL_WEIGHTS    = os.path.join(CHECKPOINT_ROOT, "combined.pth")

for d in [FRAMES_DIR, META_DIR, CHECKPOINT_ROOT]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────
IMG_SIZE          = 224
FRAMES_PER_VIDEO  = 5        # 5 frames sampled evenly over 10 s
BATCH_SIZE        = 4
NUM_WORKERS       = 2
EPOCHS            = 20
LR                = 3e-4
USE_AMP           = True
GRAD_CLIP         = 1.0
SAVE_BATCH_EVERY  = 150      # save mid-epoch checkpoint every N batches

# ── Split sizes ──────────────────────────────────────────────
TRAIN_SIZE = 3100
VAL_SIZE   = 100
TEST_SIZE  = 300

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm")

print("Config loaded.")
print(f"Train:{TRAIN_SIZE}  Val:{VAL_SIZE}  Test:{TEST_SIZE}  Frames/video:{FRAMES_PER_VIDEO}")


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Config loaded.
Train:3100  Val:100  Test:300  Frames/video:5


In [ ]:
# ============================================================
# CELL 4 — Extract Celeb-DF zip (skips if already done)
# ============================================================
if not os.path.exists(CELEBDF_EXTRACT):
    print("Extracting Celeb-DF.zip ...")
    os.makedirs(CELEBDF_EXTRACT, exist_ok=True)
    with zipfile.ZipFile(CELEBDF_ZIP, "r") as z:
        z.extractall(CELEBDF_EXTRACT)
    print("Extraction complete.")
else:
    print("Celeb-DF already extracted, skipping.")

# Print top-level structure
for item in sorted(os.listdir(CELEBDF_EXTRACT)):
    path = os.path.join(CELEBDF_EXTRACT, item)
    n = len(os.listdir(path)) if os.path.isdir(path) else "-"
    print(f"  {item}/  ({n} items)")


Extracting Celeb-DF.zip ...
Extraction complete.
  Celeb-real/  (158 items)
  Celeb-synthesis/  (795 items)
  List_of_testing_videos.txt/  (- items)
  YouTube-real/  (250 items)


In [ ]:
# ============================================================
# CELL 5 — Collect & Merge all video paths
# ============================================================
def list_videos(folder):
    videos = []
    if not os.path.isdir(folder):
        print(f"WARNING: folder not found: {folder}")
        return videos
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(VIDEO_EXTS):
                videos.append(os.path.join(root, f))
    return sorted(videos)

# DS1
ds1_real = list_videos(DS1_REAL)
ds1_fake = list_videos(DS1_FAKE)

# DS2
ds2_real = list_videos(DS2_REAL)
ds2_fake = list_videos(DS2_FAKE)

# DS3 — Celeb-DF
celeb_yt_real  = list_videos(os.path.join(CELEBDF_EXTRACT, "YouTube-real"))
celeb_real     = list_videos(os.path.join(CELEBDF_EXTRACT, "Celeb-real"))
celeb_synthesis = list_videos(os.path.join(CELEBDF_EXTRACT, "Celeb-synthesis"))

# Merge
all_real = ds1_real + ds2_real + celeb_yt_real + celeb_real
all_fake = ds1_fake + ds2_fake + celeb_synthesis

print(f"DS1  real:{len(ds1_real)}  fake:{len(ds1_fake)}")
print(f"DS2  real:{len(ds2_real)}  fake:{len(ds2_fake)}")
print(f"DS3  yt_real:{len(celeb_yt_real)}  celeb_real:{len(celeb_real)}  synth:{len(celeb_synthesis)}")
print(f"TOTAL  real:{len(all_real)}  fake:{len(all_fake)}")
assert len(all_real) > 0, "No real videos found. Check your Drive paths."
assert len(all_fake) > 0, "No fake videos found. Check your Drive paths."


DS1  real:1000  fake:1000
DS2  real:1000  fake:1000
DS3  yt_real:250  celeb_real:158  synth:795
TOTAL  real:2408  fake:2795


In [ ]:
# ============================================================
# CELL 6 — Build dataset list & train/val/test splits
# Targets: Train=3100, Val=100, Test=300  (total 3500)
# ============================================================
TOTAL_NEEDED = TRAIN_SIZE + VAL_SIZE + TEST_SIZE   # 3500
PER_CLASS    = TOTAL_NEEDED // 2                   # 1750 each

# Shuffle before selecting
random.shuffle(all_real); random.shuffle(all_fake)

# Cap to what's available
real_sel = all_real[:min(len(all_real), PER_CLASS)]
fake_sel = all_fake[:min(len(all_fake), PER_CLASS)]

if len(real_sel) < PER_CLASS:
    print(f"WARNING: only {len(real_sel)} real videos available (need {PER_CLASS}). Using all.")
if len(fake_sel) < PER_CLASS:
    print(f"WARNING: only {len(fake_sel)} fake videos available (need {PER_CLASS}). Using all.")

samples = (
    [{"video_path": p, "label": 0} for p in real_sel] +
    [{"video_path": p, "label": 1} for p in fake_sel]
)
random.shuffle(samples)

# ── Stratified split ─────────────────────────────────────────
labels_all = [s["label"] for s in samples]

# We need exactly TRAIN_SIZE train, rest split into val/test
test_val_size = VAL_SIZE + TEST_SIZE
train_s, temp_s = train_test_split(
    samples, test_size=test_val_size, stratify=labels_all, random_state=SEED
)

# Trim/pad train to TRAIN_SIZE if needed
if len(train_s) > TRAIN_SIZE:
    train_s = train_s[:TRAIN_SIZE]

temp_labels = [s["label"] for s in temp_s]
val_s, test_s = train_test_split(
    temp_s,
    test_size=TEST_SIZE,
    stratify=temp_labels,
    random_state=SEED
)
val_s  = val_s[:VAL_SIZE]
test_s = test_s[:TEST_SIZE]

print(f"Train: {len(train_s)}  Val: {len(val_s)}  Test: {len(test_s)}")
print(f"Train — Real:{sum(x['label']==0 for x in train_s)}  Fake:{sum(x['label']==1 for x in train_s)}")
print(f"Val   — Real:{sum(x['label']==0 for x in val_s)}  Fake:{sum(x['label']==1 for x in val_s)}")
print(f"Test  — Real:{sum(x['label']==0 for x in test_s)}  Fake:{sum(x['label']==1 for x in test_s)}")

# Save split metadata
for name, data in [("train", train_s), ("val", val_s), ("test", test_s)]:
    with open(os.path.join(META_DIR, f"{name}_samples.json"), "w") as f:
        json.dump(data, f, indent=2)
print("Split metadata saved.")


Train: 3100  Val: 100  Test: 300
Train — Real:1550  Fake:1550
Val   — Real:50  Fake:50
Test  — Real:150  Fake:150
Split metadata saved.


In [ ]:
# ============================================================
# CELL 7 — Face detection + COBRA perceptual enhancement
#
# COBRA algorithm: Contrast-Optimal Bilateral Resolution Amplifier
# Enhances spatial detail in face crops using bilateral filtering
# and adaptive unsharp masking WITHOUT touching high-frequency
# forensic noise patterns (operates only on low-freq structure).
# ============================================================
try:
    import mediapipe as mp
    _mp_face = mp.solutions.face_detection
    MEDIAPIPE_OK = True
    print("MediaPipe available.")
except ImportError:
    MEDIAPIPE_OK = False
    print("MediaPipe not found — using Haar cascade fallback.")

HAAR = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def cobra_enhance(frame_bgr: np.ndarray) -> np.ndarray:
    """
    COBRA: Contrast-Optimal Bilateral Resolution Amplifier
    -------------------------------------------------------
    1. Bilateral filter → smooth noise while preserving edges (low-freq).
    2. High-frequency residual = original − bilateral.
    3. Enhance only the smooth layer with CLAHE contrast.
    4. Recombine: smooth_enh + residual  → quality up, fingerprints intact.
    """
    # Step 1 — bilateral (d=5 keeps it fast on GPU colab)
    smooth = cv2.bilateralFilter(frame_bgr, d=5, sigmaColor=40, sigmaSpace=40)

    # Step 2 — high-freq residual (preserves forensic artifacts)
    residual = frame_bgr.astype(np.int16) - smooth.astype(np.int16)

    # Step 3 — CLAHE on luminance channel only
    lab = cv2.cvtColor(smooth, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enh = clahe.apply(l)
    smooth_enh = cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

    # Step 4 — recombine
    out = np.clip(smooth_enh.astype(np.int16) + residual, 0, 255).astype(np.uint8)
    return out


def detect_and_crop_face(frame_bgr: np.ndarray, target: int = 224) -> np.ndarray:
    """Detect face, apply COBRA enhancement, return RGB crop."""
    h, w = frame_bgr.shape[:2]

    # ── MediaPipe path ──────────────────────────────────────
    if MEDIAPIPE_OK:
        try:
            with _mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.4) as det:
                rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                res = det.process(rgb)
                if res.detections:
                    d = max(res.detections, key=lambda x: x.score[0])
                    bb = d.location_data.relative_bounding_box
                    pad = 0.15
                    x1 = max(0, int((bb.xmin - pad * bb.width) * w))
                    y1 = max(0, int((bb.ymin - pad * bb.height) * h))
                    x2 = min(w, int((bb.xmin + (1 + pad) * bb.width) * w))
                    y2 = min(h, int((bb.ymin + (1 + pad) * bb.height) * h))
                    crop = frame_bgr[y1:y2, x1:x2]
                    if crop.size > 0:
                        crop = cobra_enhance(crop)
                        crop = cv2.resize(crop, (target, target))
                        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        except Exception:
            pass

    # ── Haar fallback ───────────────────────────────────────
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = HAAR.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
    if len(faces) > 0:
        x, y, fw, fh = sorted(faces, key=lambda f: f[2]*f[3])[-1]
        pad_x, pad_y = int(fw * 0.15), int(fh * 0.15)
        x1 = max(0, x - pad_x); y1 = max(0, y - pad_y)
        x2 = min(w, x + fw + pad_x); y2 = min(h, y + fh + pad_y)
        crop = frame_bgr[y1:y2, x1:x2]
    else:
        # centre crop fallback
        crop = frame_bgr[h//8:7*h//8, w//8:7*w//8]

    crop = cobra_enhance(crop)
    crop = cv2.resize(crop, (target, target))
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def evenly_spaced(n_frames: int, k: int) -> list:
    if n_frames <= 0: return []
    if n_frames < k:
        idx = list(range(n_frames))
        while len(idx) < k: idx.append(n_frames - 1)
        return idx
    return np.linspace(0, n_frames - 1, k, dtype=int).tolist()


print("COBRA + face detector ready.")


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


MediaPipe available.
COBRA + face detector ready.


In [ ]:
# ============================================================
# CELL 8 — Extract frames from videos (5 frames / 10 s window)
# Saves JPGs and writes metadata JSON per split
# ============================================================
def extract_split(sample_list, split_name, frames_k=FRAMES_PER_VIDEO):
    split_dir = os.path.join(FRAMES_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    meta = []

    for item in tqdm(sample_list, desc=f"Extracting [{split_name}]"):
        vpath = item["video_path"]
        label = item["label"]
        vid_id = os.path.splitext(os.path.basename(vpath))[0]
        out_dir = os.path.join(split_dir, f"{label}_{vid_id[:60]}")
        os.makedirs(out_dir, exist_ok=True)

        cap = cv2.VideoCapture(vpath)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Sample 5 frames from first 10 seconds
        max_frame = min(total, max(1, int(fps * 10)))
        indices = set(evenly_spaced(max_frame, frames_k))

        saved, fid = [], 0
        while True:
            ret, frame = cap.read()
            if not ret or fid > max_frame: break
            if fid in indices:
                try:
                    face = detect_and_crop_face(frame, IMG_SIZE)
                    sp = os.path.join(out_dir, f"f{len(saved):02d}.jpg")
                    cv2.imwrite(sp, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                    saved.append(sp)
                except Exception as e:
                    pass
            fid += 1
        cap.release()

        if not saved: continue

        # Pad if fewer than k frames extracted
        while len(saved) < frames_k:
            dst = os.path.join(out_dir, f"f{len(saved):02d}.jpg")
            shutil.copy(saved[-1], dst)
            saved.append(dst)

        meta.append({"video_path": vpath, "label": label,
                     "frame_paths": saved[:frames_k]})

    mpath = os.path.join(META_DIR, f"{split_name}_frames.json")
    with open(mpath, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"{split_name}: {len(meta)} videos processed → {mpath}")
    return meta


In [ ]:
# ============================================================
# CELL 9 — Run frame extraction for all splits
# This is the longest step. Colab Pro GPU not needed here.
# ============================================================
train_meta = extract_split(train_s, "train")
val_meta   = extract_split(val_s,   "val")
test_meta  = extract_split(test_s,  "test")
print(f"Done. train:{len(train_meta)} val:{len(val_meta)} test:{len(test_meta)}")


Extracting [train]:   0%|          | 0/3100 [00:00<?, ?it/s]

train: 3100 videos processed → /content/deepfake_meta/train_frames.json


Extracting [val]:   0%|          | 0/100 [00:00<?, ?it/s]

val: 100 videos processed → /content/deepfake_meta/val_frames.json


Extracting [test]:   0%|          | 0/300 [00:00<?, ?it/s]

test: 300 videos processed → /content/deepfake_meta/test_frames.json
Done. train:3100 val:100 test:300


In [ ]:
# ============================================================
# CELL 10 — Transforms & Dataset class
# ============================================================
TRAIN_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25,
                           saturation=0.15, hue=0.05),
    transforms.RandomGrayscale(p=0.04),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


def make_fft_tensor(pil_img: Image.Image) -> torch.Tensor:
    """Convert PIL image to FFT magnitude image tensor (forensic frequency map)."""
    gray = np.array(pil_img.convert("L"), dtype=np.float32)
    fft_shift = np.fft.fftshift(np.fft.fft2(gray))
    mag = np.log(np.abs(fft_shift) + 1.0)
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    mag_u8 = (mag * 255).astype(np.uint8)
    fft_pil = Image.fromarray(np.stack([mag_u8]*3, axis=-1))
    return EVAL_TF(fft_pil)  # always deterministic


class DeepfakeDataset(Dataset):
    def __init__(self, meta, augment=False):
        self.meta    = meta
        self.augment = augment
        self.rgb_tf  = TRAIN_TF if augment else EVAL_TF

    def __len__(self): return len(self.meta)

    def __getitem__(self, idx):
        item = self.meta[idx]
        fps  = item["frame_paths"]
        label = int(item["label"])

        rgb_frames, fft_frames = [], []
        for fp in fps:
            pil = Image.open(fp).convert("RGB")
            rgb_frames.append(self.rgb_tf(pil))
            fft_frames.append(make_fft_tensor(pil))

        return (torch.stack(rgb_frames),   # [T,3,H,W]
                torch.stack(fft_frames),   # [T,3,H,W]
                torch.tensor(label, dtype=torch.long))


# ── DataLoaders ─────────────────────────────────────────────
train_ds = DeepfakeDataset(train_meta, augment=True)
val_ds   = DeepfakeDataset(val_meta,   augment=False)
test_ds  = DeepfakeDataset(test_meta,  augment=False)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS>0))
val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS>0))
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS>0))

print(f"Loaders ready — train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)} batches")


Loaders ready — train:775 val:25 test:75 batches


In [ ]:
# Remove old torch
!pip uninstall -y torch torchvision torchaudio -q

# Install latest compatible version (supports newer GPUs like G4)
!pip install -q torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.7.1+cu118
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
# ============================================================
# CELL 11 — Model: MesoNet + EfficientNet-B4 + FFT + Temporal BiLSTM
# ============================================================

# ── Branch 1: MesoNet (lightweight, artefact-focused) ────────
class MesoNetBranch(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  8, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(8),  nn.MaxPool2d(2),
            nn.Conv2d(8,  8, 5, padding=2), nn.ReLU(), nn.BatchNorm2d(8),  nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 5, padding=2), nn.ReLU(), nn.BatchNorm2d(16), nn.MaxPool2d(2),
            nn.Conv2d(16,16, 5, padding=2), nn.ReLU(), nn.BatchNorm2d(16),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*4*4, 256), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(256, out_dim)
        )
    def forward(self, x): return self.fc(self.features(x))


# ── Branch 2: EfficientNet-B4 (stronger backbone) ────────────
class EfficientNetBranch(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = EfficientNet.from_pretrained("efficientnet-b4")
        in_feat = self.backbone._fc.in_features
        self.backbone._fc = nn.Identity()
        self.proj = nn.Sequential(
            nn.Linear(in_feat, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, out_dim)
        )
    def forward(self, x): return self.proj(self.backbone(x))


# ── Branch 3: FFT frequency-domain branch ────────────────────
class FFTBranch(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(64*4*4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, out_dim)
        )
    def forward(self, x): return self.net(x)


# ── Fusion: Temporal Conv → BiLSTM → Classifier ──────────────
class CombinedDeepfakeModel(nn.Module):
    def __init__(self, num_classes=2, lstm_hidden=384):
        super().__init__()
        self.meso = MesoNetBranch(out_dim=128)
        self.eff  = EfficientNetBranch(out_dim=512)
        self.fft  = FFTBranch(out_dim=128)

        fused = 128 + 512 + 128   # 768

        self.temporal = nn.Sequential(
            nn.Conv1d(fused, fused, 3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(fused),
            nn.Conv1d(fused, fused, 3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(fused)
        )

        self.lstm = nn.LSTM(
            input_size=fused, hidden_size=lstm_hidden,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.3
        )

        self.attn = nn.Linear(lstm_hidden * 2, 1)  # attention over time

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, rgb_seq, fft_seq):
        # rgb_seq / fft_seq : [B, T, C, H, W]
        B, T, C, H, W = rgb_seq.shape
        r = rgb_seq.view(B*T, C, H, W)
        f = fft_seq.view(B*T, C, H, W)

        feat = torch.cat([self.meso(r), self.eff(r), self.fft(f)], dim=1)  # [B*T, F]
        feat = feat.view(B, T, -1)                                           # [B, T, F]

        # Temporal conv
        tc = feat.permute(0,2,1)        # [B,F,T]
        tc = self.temporal(tc)
        tc = tc.permute(0,2,1)          # [B,T,F]

        # BiLSTM
        lstm_out, _ = self.lstm(tc)     # [B,T,2H]

        # Attention pooling (fixes constant-output problem from last-step only)
        attn_w = torch.softmax(self.attn(lstm_out), dim=1)  # [B,T,1]
        ctx    = (attn_w * lstm_out).sum(dim=1)              # [B,2H]

        return self.head(ctx)


model = CombinedDeepfakeModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
for module in model.modules():
    if isinstance(module, torch.nn.LSTM):
        module.flatten_parameters = lambda: None
print(f"Model on {DEVICE} | Trainable params: {total_params/1e6:.1f}M")


Loaded pretrained weights for efficientnet-b4


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ============================================================
# CELL 12 — Optimizer, Scheduler, Loss
# ============================================================
# Class-balanced loss (handles any imbalance automatically)
train_labels = [s["label"] for s in train_meta]
n_real = train_labels.count(0)
n_fake = train_labels.count(1)
w_real = len(train_labels) / (2.0 * max(n_real, 1))
w_fake = len(train_labels) / (2.0 * max(n_fake, 1))
class_weights = torch.tensor([w_real, w_fake], dtype=torch.float32).to(DEVICE)
print(f"Class weights — real:{w_real:.3f}  fake:{w_fake:.3f}")

criterion  = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1, anneal_strategy="cos"
)
scaler     = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Optimizer: AdamW + OneCycleLR ready.")


Class weights — real:1.000  fake:1.000
Optimizer: AdamW + OneCycleLR ready.


In [ ]:
# ============================================================
# CELL 13 — Training helpers: one epoch, checkpoint utils
# ============================================================
def save_ckpt(state, path):
    torch.save(state, path)


def run_epoch(model, loader, optimizer=None, train=True, epoch=0):
    model.train() if train else model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"{'Train' if train else 'Val/Test'} E{epoch}", leave=True)

    for b_idx, (rgb, fft_b, lbl) in enumerate(pbar):
        rgb   = rgb.to(DEVICE, non_blocking=True)
        fft_b = fft_b.to(DEVICE, non_blocking=True)
        lbl   = lbl.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(rgb, fft_b)
                loss   = criterion(logits, lbl)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                # Mid-epoch checkpoint
                if (b_idx + 1) % SAVE_BATCH_EVERY == 0:
                    save_ckpt({
                        "epoch": epoch, "batch": b_idx+1,
                        "model_state": model.state_dict(),
                        "opt_state": optimizer.state_dict()
                    }, os.path.join(CHECKPOINT_ROOT,
                                    f"ep{epoch:02d}_b{b_idx+1:05d}.pth"))

        probs  = torch.softmax(logits.detach(), dim=1)[:,1]
        preds  = torch.argmax(logits.detach(), dim=1)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(lbl.cpu().numpy().tolist())
        total_loss += loss.item()

        acc = accuracy_score(all_labels, all_preds)
        f1  = f1_score(all_labels, all_preds, zero_division=0)
        pbar.set_postfix(loss=f"{total_loss/(b_idx+1):.4f}",
                         acc=f"{acc:.4f}", f1=f"{f1:.4f}")

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return avg_loss, acc, f1, auc, all_preds, all_labels

print("Training helpers ready.")


Training helpers ready.


In [ ]:
# ============================================================
# CELL 14 — Training Loop (20 epochs, checkpoints per epoch)
# ============================================================
history = []
best_val_f1  = -1.0
best_val_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc, t_f1, t_auc, _, _ = run_epoch(
        model, train_loader, optimizer, train=True, epoch=epoch)

    with torch.no_grad():
        v_loss, v_acc, v_f1, v_auc, v_preds, v_labels = run_epoch(
            model, val_loader, optimizer=None, train=False, epoch=epoch)

    lr_now = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch:02d}/{EPOCHS}")
    print(f"  Train  loss:{t_loss:.4f}  acc:{t_acc:.4f}  f1:{t_f1:.4f}  auc:{t_auc:.4f}")
    print(f"  Val    loss:{v_loss:.4f}  acc:{v_acc:.4f}  f1:{v_f1:.4f}  auc:{v_auc:.4f}  lr:{lr_now:.2e}")

    row = dict(epoch=epoch,
               t_loss=t_loss, t_acc=t_acc, t_f1=t_f1,  t_auc=t_auc,
               v_loss=v_loss, v_acc=v_acc, v_f1=v_f1,  v_auc=v_auc)
    history.append(row)

    # Per-epoch checkpoint
    ckpt_path = os.path.join(CHECKPOINT_ROOT, f"epoch_{epoch:02d}.pth")
    save_ckpt({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "opt_state": optimizer.state_dict(),
        "history": history
    }, ckpt_path)
    print(f"  Checkpoint saved → {ckpt_path}")

    # Best model
    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        save_ckpt({"epoch": epoch, "model_state": model.state_dict(),
                   "history": history},
                  os.path.join(CHECKPOINT_ROOT, "best_model.pth"))
        print(f"  ✓ Best model updated (val_f1={best_val_f1:.4f})")

print("\nTraining complete.")
pd.DataFrame(history)


Train E1:   0%|          | 0/775 [00:00<?, ?it/s]

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ============================================================
# CELL 15 — Save final combined weights as combined.pth
# ============================================================
torch.save(model.state_dict(), FINAL_WEIGHTS)
print(f"Final weights saved → {FINAL_WEIGHTS}")

# Also save full training history
hist_path = os.path.join(CHECKPOINT_ROOT, "training_history.json")
with open(hist_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"Training history saved → {hist_path}")


In [ ]:
# ============================================================
# CELL 16 — Load best model & evaluate on test set
# ============================================================
best_ckpt = torch.load(os.path.join(CHECKPOINT_ROOT, "best_model.pth"),
                        map_location=DEVICE)
model.load_state_dict(best_ckpt["model_state"])
model.eval()
print(f"Loaded best model from epoch {best_ckpt['epoch']}")

_, te_acc, te_f1, te_auc, te_preds, te_labels = run_epoch(
    model, test_loader, train=False, epoch=0)

te_prec = precision_score(te_labels, te_preds, zero_division=0)
te_rec  = recall_score(te_labels, te_preds, zero_division=0)

print("\n" + "="*55)
print("TEST SET RESULTS")
print("="*55)
print(f"  Accuracy  : {te_acc*100:.2f}%")
print(f"  F1 Score  : {te_f1:.4f}")
print(f"  AUC-ROC   : {te_auc:.4f}")
print(f"  Precision : {te_prec:.4f}")
print(f"  Recall    : {te_rec:.4f}")
print("="*55)
print("\nClassification Report:")
print(classification_report(te_labels, te_preds,
                             target_names=["Real","Fake"], zero_division=0))
print("Confusion Matrix:")
cm = confusion_matrix(te_labels, te_preds)
print(cm)
tn,fp,fn,tp = cm.ravel()
print(f"  TP:{tp}  TN:{tn}  FP:{fp}  FN:{fn}")
print(f"  Sensitivity (Real detected): {tn/(tn+fp+1e-8)*100:.1f}%")
print(f"  Specificity (Fake detected): {tp/(tp+fn+1e-8)*100:.1f}%")


In [ ]:
# ============================================================
# CELL 17 — Load combined.pth for inference (run in new sessions)
# ============================================================
# Make sure CELL 3 (imports/config) and CELL 11 (model definition)
# are run first, then run this cell to restore weights.

model = CombinedDeepfakeModel().to(DEVICE)
state = torch.load(FINAL_WEIGHTS, map_location=DEVICE)
model.load_state_dict(state)
model.eval()
print(f"Weights loaded from {FINAL_WEIGHTS}")
print("Model is ready for inference.")


In [ ]:
# ============================================================
# CELL 18 — Inference: predict any video file
# ============================================================
def predict_video(video_path: str,
                  frames_k: int = FRAMES_PER_VIDEO,
                  threshold: float = 0.45) -> dict:
    """
    Predict whether a video is Real or Fake.

    Parameters
    ----------
    video_path : str  — local path (Drive or /content)
    frames_k   : int  — number of frames to sample (default 5)
    threshold  : float — fake probability threshold (default 0.45)

    Returns
    -------
    dict with prediction, probabilities, confidence
    """
    assert os.path.isfile(video_path), f"File not found: {video_path}"

    tmp_dir = "/content/_infer_tmp"
    if os.path.exists(tmp_dir): shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_f  = min(total, max(1, int(fps * 10)))
    idxs   = set(evenly_spaced(max_f, frames_k))

    saved, fid = [], 0
    while True:
        ret, frame = cap.read()
        if not ret or fid > max_f: break
        if fid in idxs:
            try:
                face = detect_and_crop_face(frame, IMG_SIZE)
                sp = os.path.join(tmp_dir, f"f{len(saved):02d}.jpg")
                cv2.imwrite(sp, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                saved.append(sp)
            except Exception:
                pass
        fid += 1
    cap.release()
    assert saved, "No frames could be extracted from the video."

    while len(saved) < frames_k:
        dst = os.path.join(tmp_dir, f"f{len(saved):02d}.jpg")
        shutil.copy(saved[-1], dst); saved.append(dst)
    saved = saved[:frames_k]

    # Build tensors
    rgb_list, fft_list = [], []
    for sp in saved:
        pil = Image.open(sp).convert("RGB")
        rgb_list.append(EVAL_TF(pil))
        fft_list.append(make_fft_tensor(pil))

    rgb_t = torch.stack(rgb_list).unsqueeze(0).to(DEVICE)
    fft_t = torch.stack(fft_list).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(rgb_t, fft_t)
            probs  = torch.softmax(logits, dim=1)[0]

    real_p = probs[0].item()
    fake_p = probs[1].item()
    label  = "FAKE" if fake_p >= threshold else "REAL"
    conf   = fake_p if label == "FAKE" else real_p

    print("=" * 50)
    print("  DEEPFAKE DETECTION RESULT")
    print("=" * 50)
    print(f"  File       : {os.path.basename(video_path)}")
    print(f"  Prediction : {label}")
    print(f"  Confidence : {conf*100:.2f}%")
    print(f"  Real prob  : {real_p*100:.2f}%")
    print(f"  Fake prob  : {fake_p*100:.2f}%")
    print(f"  Frames used: {len(saved)}")
    print("=" * 50)
    return {"prediction": label, "confidence": conf,
            "real_prob": real_p, "fake_prob": fake_p}


print("predict_video() ready.")
print('Usage: predict_video("/content/drive/MyDrive/my_video.mp4")')


In [ ]:
# ============================================================
# CELL 19 — Demo: upload & predict
# ============================================================
from google.colab import files
import matplotlib.pyplot as plt

print("Upload a video file:")
uploaded = files.upload()

for fname in uploaded:
    print(f"\nProcessing: {fname}")
    result = predict_video(fname)

    # Show extracted frames
    tmp_dir = "/content/_infer_tmp"
    frame_files = sorted(os.listdir(tmp_dir))[:5]
    if frame_files:
        fig, axes = plt.subplots(1, len(frame_files), figsize=(15,3))
        if len(frame_files) == 1: axes = [axes]
        for ax, fn in zip(axes, frame_files):
            img = Image.open(os.path.join(tmp_dir, fn))
            ax.imshow(img); ax.axis("off")
            ax.set_title(f"Frame")
        plt.suptitle(f"Prediction: {result['prediction']} ({result['confidence']*100:.1f}%)",
                     fontsize=14, fontweight="bold",
                     color="red" if result["prediction"]=="FAKE" else "green")
        plt.tight_layout()
        plt.show()
